# TEMPLATE — Tên kịch bản mới

Notebook này là hồ sơ phân tích có thể tái lập. Mỗi CSV tương ứng một lần chạy;
notebook so sánh nhiều lần chạy và nhiều cấu hình.

> **Quy tắc:** Không chỉnh sửa dữ liệu trong `data/raw/`. Nếu một lần chạy không
hợp lệ, giữ lại tệp và ghi rõ lý do.

## 1. Nhận dạng thử nghiệm

Điền người thực hiện, phiên bản phần mềm, cấu hình phần cứng và các mã lần chạy
trước khi phân tích.

In [ ]:
MA_THU = "TEMPLATE"
TEN_THU = "Tên kịch bản mới"
MA_CAU_HINH_GOC = "CHUA_DIEN"
MA_CAU_HINH_MOI = "CHUA_DIEN"
CAC_RUN_ID: list[str] = []
DUNG_SAI_HEADING_DEG = 1.0
THOI_GIAN_GIU_DUNG_SAI_S = 0.5

## 2. Câu hỏi kỹ thuật

**Câu hỏi:** Thay đổi đang xét ảnh hưởng thế nào đến hiệu năng robot?

**Yêu cầu đo được:** Điền một yêu cầu có đơn vị, dung sai và cách đo.

**Giả thuyết:** Điền dự đoán trước khi xem kết quả.

### Biến thử nghiệm

- Biến độc lập:
- Chỉ số phụ thuộc:
- Biến được giữ cố định: pin, bề mặt, tải, tâm khối lượng, bánh xe, vị trí xuất phát.

## 3. Chuẩn bị môi trường phân tích

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

GOC = Path.cwd().resolve()
if GOC.name == "notebooks":
    GOC = GOC.parent
sys.path.insert(0, str(GOC / "src"))

from fgc_analysis.io import tai_cac_lan_chay, tai_tom_tat
from fgc_analysis.metrics import (
    do_lech_mm_m,
    thoi_gian_on_dinh,
    thoi_gian_tang_10_90,
    tong_hop_lan_chay,
)
from fgc_analysis.plots import (
    cai_dat_giao_dien,
    ve_dien,
    ve_heading,
    ve_pid,
    ve_van_toc,
)
from fgc_analysis.validation import kiem_tra_du_lieu

cai_dat_giao_dien()
pd.set_option("display.max_columns", 100)
print(f"Thư mục project: {GOC}")

## 4. Nạp dữ liệu gốc

In [ ]:
THU_MUC_DU_LIEU = GOC / "data" / "raw" / MA_THU
du_lieu = tai_cac_lan_chay(THU_MUC_DU_LIEU)

if CAC_RUN_ID and not du_lieu.empty:
    du_lieu = du_lieu[du_lieu["run_id"].isin(CAC_RUN_ID)].copy()

if du_lieu.empty:
    display(Markdown(
        f"**Chưa có dữ liệu.** Hãy chép CSV vào `{THU_MUC_DU_LIEU}` "
        "rồi chạy lại notebook."
    ))
else:
    print(f"Đã nạp {len(du_lieu):,} mẫu từ {du_lieu['run_id'].nunique()} lần chạy.")
    display(du_lieu.head())

## 5. Kiểm tra chất lượng dữ liệu

In [ ]:
bao_cao_kiem_tra = kiem_tra_du_lieu(du_lieu)
display(bao_cao_kiem_tra)

co_loi = (
    not bao_cao_kiem_tra.empty
    and bao_cao_kiem_tra["muc_do"].eq("LỖI").any()
)
if co_loi:
    display(Markdown(
        "> **Dừng kết luận:** Dữ liệu có lỗi. Hãy giải thích hoặc sửa quy trình "
        "ghi log trước khi dùng các chỉ số."
    ))

## 6. Chọn đoạn đo

In [ ]:
doan_do = du_lieu.copy()
if not doan_do.empty and "recording" in doan_do:
    gia_tri = doan_do["recording"]
    if gia_tri.dtype == bool:
        doan_do = doan_do[gia_tri].copy()
    else:
        doan_do = doan_do[
            gia_tri.astype(str).str.lower().isin({"true", "1", "yes"})
        ].copy()

if not doan_do.empty:
    print(f"Đoạn phân tích có {len(doan_do):,} mẫu.")

## 7. Chỉ số tự động

In [ ]:
bang_chi_so = tong_hop_lan_chay(doan_do)
display(bang_chi_so)

## 8. Phân tích riêng cho kịch bản

Bổ sung phép tính riêng của kịch bản tại đây. Đưa hàm tái sử dụng vào `src/fgc_analysis/`.

## 9. Biểu đồ chuẩn

In [ ]:
if not doan_do.empty:
    for run_id, mot_run in doan_do.groupby("run_id", sort=False):
        ve_van_toc(mot_run, f"{run_id} — vận tốc mục tiêu và thực tế")
        plt.show()
        ve_heading(mot_run, f"{run_id} — heading")
        plt.show()

        if {"heading_p", "heading_i", "heading_d", "heading_total"}.issubset(mot_run.columns):
            ve_pid(mot_run, f"{run_id} — thành phần hiệu chỉnh heading")
            plt.show()

        if "battery_v" in mot_run:
            ve_dien(mot_run, f"{run_id} — điện")
            plt.show()

## 10. Dữ liệu đo bên ngoài

In [ ]:
tom_tat = tai_tom_tat(GOC / "data" / "run_summary.csv")
if not tom_tat.empty and "test_id" in tom_tat:
    tom_tat = tom_tat[tom_tat["test_id"].eq(MA_THU)].copy()
display(tom_tat)

if (
    not tom_tat.empty
    and {"lateral_offset_mm", "actual_distance_mm"}.issubset(tom_tat.columns)
):
    tom_tat["drift_mm_per_m"] = tom_tat.apply(
        lambda hang: do_lech_mm_m(
            hang["lateral_offset_mm"],
            hang["actual_distance_mm"],
        ),
        axis=1,
    )
    display(tom_tat[["run_id", "actual_distance_mm", "lateral_offset_mm", "drift_mm_per_m"]])

## 11. Kết quả và yêu cầu

Điền bảng sau sau khi chỉ số đã được kiểm tra:

| Chỉ số | Mốc gốc | Cấu hình mới | Mục tiêu | Kết quả |
| --- | ---: | ---: | ---: | --- |
| Sai số quãng đường | | | | |
| Độ lệch ngang | | | | |
| RMSE heading | | | | |
| RMSE vận tốc | | | | |
| Dòng điện đỉnh | | | | |
| Sụt áp | | | | |

Không đánh dấu **ĐẠT** nếu thiếu phép đo bên ngoài bắt buộc.

## 12. Quan sát và bất thường

- Trượt bánh:
- Rung hoặc tiếng ồn:
- Va chạm/gờ sàn:
- Lỗi thao tác:
- Phần cứng lỏng:
- Lần chạy bị loại và lý do:

## 13. Kết luận kỹ thuật

### Bằng chứng trả lời câu hỏi


### Giả thuyết được ủng hộ hay bác bỏ?


### Quyết định

- [ ] Giữ thay đổi
- [ ] Sửa rồi thử lại
- [ ] Hoàn tác
- [ ] Thay đổi cơ khí
- [ ] Chấp nhận giới hạn đã đo

### Hành động tiếp theo

- Người phụ trách:
- Mã thử nghiệm tiếp theo:
- Câu hỏi tiếp theo: